# Data Preparation
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Daniel Alejandro Castillo Martin

---

This notebook prepares the Enron spam dataset for the midterm neural network architecture comparison. It documents data-quality checks, duplicate handling, reproducible partitioning, tokenization, vocabulary construction, and batch verification.

The vocabulary is fitted exclusively on training text. Prepared partitions, the vocabulary, and preprocessing metadata are saved for reuse by all the experiments.

## 1. Initial Configuration

### 1.1 Import, Paths and Device Configuration

In [5]:
# Import required libraries
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import torch

from torch import nn
from warnings import filterwarnings

# Ignore warnings
filterwarnings('ignore')

In [6]:
# Locate the repository from either its root or the notebooks folder to access local modules
current_dir = Path.cwd().resolve()

PROJECT_ROOT = (current_dir if (current_dir / 'src').is_dir() else current_dir.parent)

if not (PROJECT_ROOT / 'src').is_dir():raise FileNotFoundError('Open the notebook from the repository root or notebooks folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [7]:
pip install datasets 

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Import local modules
from src.reproducibility import set_seed
from src.data import load_enron_dataset, audit_dataset, prepare_splits, load_prepared_data, save_prepared_data
from src.text import tokenize, build_vocabulary, EncodedTextDataset, make_data_loader, measure_token_lengths

In [9]:
# Import constants from config.py
from src.config import DATASET_ID, DATASET_REVISION, SPLIT_SEED, TRAINING_SEEDS, MAX_VOCAB_SIZE, MIN_FREQUENCY, MAX_LENGTH, BATCH_SIZE, VALIDATION_FRACTION

In [10]:
# Set device dynamically to support CPU and GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print environment details
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Python: 3.12.14
PyTorch: 2.14.0+cpu
Device: cpu


### 1.2 Reproducibility

The data split uses a fixed seed so that every architecture is evaluated on the same examples. Separate training seeds are used to measure variation caused by initialization and training randomness.

Each experiment resets the random generators before creating the model and training data loader. Seeds improve repeatability, although identical results are not guaranteed across different hardware or software versions.

In [11]:
# Set seeds for reproducibility, constant across experiments
set_seed(TRAINING_SEEDS[0])

print(f'Split seed: {SPLIT_SEED}')
print(f'Training seeds: {TRAINING_SEEDS}')

Split seed: 42
Training seeds: (42, 123, 2026)


In [13]:
from pathlib import Path
from datasets import load_dataset

# Download the exact dataset from Hugging Face
raw_dataset = load_dataset(
    "SetFit/enron_spam",
    cache_dir="C:/hf_cache",
    revision=DATASET_REVISION
)

print(raw_dataset)

Repo card metadata block was not found. Setting CardData to empty.
Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 38685.88 examples/s]

DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})


In [14]:
# Check available dataset splits
print(raw_dataset.keys())

# Check the training data columns
print(raw_dataset["train"].column_names)

# Display one training example
print(raw_dataset["train"][0])

dict_keys(['train', 'test'])
['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
{'message_id': 33214, 'text': 'any software just for 15 $ - 99 $ understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'label': 1, 'label_text': 'spam', 'subject': 'any software just for 15 $ - 99 $', 'message': 'understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'date': datetime.datetime(2005, 6, 18, 0, 0)}


## 2. Data Loading and Preparation

### 2.1. Load the Source Dataset

The Enron spam dataset is loaded with its original partitions preserved. Its structure is inspected before defining the shared experimental splits. No vocabulary fitting or model training is performed at this stage.

In [15]:
# Load the dataset
raw_dataset = load_enron_dataset(
    cache_dir=Path("C:/hf_cache"),
    revision=DATASET_REVISION
)

print(raw_dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})


In [16]:
# Inspect the dataset structure for training and test partitions
for split_name, split_data in raw_dataset.items():
    print(f'Partition: {split_name}')
    print(f'Rows: {len(split_data):,}')
    print(f'Columns: {split_data.column_names}')
    print(f'Features: {split_data.features}\n')

Partition: train
Rows: 31,716
Columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
Features: {'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}

Partition: test
Rows: 2,000
Columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
Features: {'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}



### 2.2. Source Data Quality Audit

The source partitions are checked for invalid text, invalid labels, duplicate emails, conflicting labels, and train/test text overlap. Duplicate detection normalizes case and whitespace without changing the model input. No records are removed during this audit.

In [17]:
# Perform data quality inspection
quality_summary, shared_text_count = audit_dataset(raw_dataset)
quality_summary

,split,rows,missing_or_nonstring_text,empty_text,invalid_labels,duplicate_extra_rows,conflicting_label_groups
0,train,31716,0,51,0,3137,0
1,test,2000,0,0,0,19,0


In [18]:
# Show distinct shared texts
print('Distinct normalized texts shared by source train and test:', shared_text_count)

Distinct normalized texts shared by source train and test: 372


### 2.3. Clean Data and Create Validation Split

Invalid and empty records are removed, and normalized duplicate texts are reduced to their first occurrence within each source partition. Training records overlapping the source test partition are excluded.

The remaining training records are divided into training and validation partitions using a fixed seed and stratification by label. Normalized text overlap is checked across all resulting partitions.

In [19]:
# Prepare and clean train and test splits
splits, cleaning_summary = prepare_splits(raw_dataset, split_seed = SPLIT_SEED, validation_fraction = 0.2)
cleaning_summary

,source_split,original_rows,invalid_or_empty_removed,duplicate_rows_removed,train_test_overlap_removed
0,train,31716,51,3137,372
1,test,2000,0,19,0


In [20]:
# Observe split sizes 
split_sizes = pd.DataFrame([{'split': name, 'rows': len(dataset)}for name, dataset in splits.items()])
split_sizes

,split,rows
0,train,22524
1,validation,5632
2,test,1981


### 2.4. Tokenization and Training Vocabulary

Text is lowercased and split into word tokens and individual punctuation marks. A shared vocabulary is constructed exclusively from the training partition, with reserved tokens for padding and unknown words.

The initial configuration limits the vocabulary to 10,000 entries and requires a minimum training frequency of two. These settings must remain consistent across architectures.

In [21]:
# Fit the vocabulary on training text only
vocabulary = build_vocabulary(splits['train']['text'], max_vocab_size = MAX_VOCAB_SIZE, min_frequency = MIN_FREQUENCY)

# Try a short message that mixes case and punctuation
example_text = 'Claim your FREE prize!!!'

# Show how the shared tokenizer splits the example
print('Example tokens:', tokenize(example_text))
# Map each token to an ID, using 1 when the token is unknown
print('Example token IDs:',[vocabulary.get(token, 1) for token in tokenize(example_text)])
# See how many tokens were kept, including pad and unknown
print(f'Vocabulary size: {len(vocabulary)}')

Example tokens: ['claim', 'your', 'free', 'prize', '!', '!', '!']
Example token IDs: [743, 31, 190, 1013, 34, 34, 34]
Vocabulary size: 10000


### 2.5. Encoding and Batch Preparation

Emails are encoded using the shared training vocabulary and truncated to a maximum of 256 tokens. Unknown tokens receive ID 1.

Batches are right-padded with ID 0 to the longest sequence in the batch. Original sequence lengths are retained so models can distinguish real tokens from padding. Only training batches are shuffled.

In [22]:
# Prepare development partitions. Test encoding is deferred.
encoded_datasets = {
    name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH) for name in ('train', 'validation')
    }

# Shuffle training batches with the first training seed
train_loader = make_data_loader(encoded_datasets['train'], batch_size = BATCH_SIZE, shuffle = True, seed = TRAINING_SEEDS[0])

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size=BATCH_SIZE, shuffle=False, seed=TRAINING_SEEDS[0],)

# Pull one validation batch to inspect the tensors
example_batch = next(iter(validation_loader))

# Check padded token, length, and label shapes
print('Token shape:', example_batch['tokens'].shape)
print('Length shape:', example_batch['lengths'].shape)
print('Label shape:', example_batch['labels'].shape)

# Lengths store the real token count before padding
print('First five lengths:', example_batch['lengths'][:5])

Token shape: torch.Size([64, 256])
Length shape: torch.Size([64])
Label shape: torch.Size([64])
First five lengths: tensor([193, 256, 256, 256, 256])


In [23]:
# Verify padding and token length
tokens = example_batch['tokens']
lengths = example_batch['lengths']
labels = example_batch['labels']

# Build a column index for every position in the batch
positions = torch.arange(tokens.shape[1]).unsqueeze(0)
# Mark positions at or past each real length as padding
padding_mask = positions >= lengths.unsqueeze(1)

# Padding positions must hold the padding ID
assert torch.all(tokens[padding_mask] == 0)
# Content positions must not use the padding ID
assert torch.all(tokens[~padding_mask] != 0)
# Token IDs must stay inside the vocabulary range
assert tokens.min().item() >= 0
assert tokens.max().item() < len(vocabulary)
# Labels must be binary
assert torch.all((labels == 0) | (labels == 1))

print('Batch padding, vocabulary IDs, and labels verified.')

Batch padding, vocabulary IDs, and labels verified.


### 2.6. Preprocessing Verification

Batch padding and token IDs are checked for consistency. Original training token lengths are summarized to quantify the effect of the shared sequence-length limit. Training and validation class proportions and label meanings are also verified.

In [24]:
# Count tokens in each training email before truncation
training_lengths = np.asarray(measure_token_lengths(splits['train']['text']))

# Summarize the length distribution, including the upper percentiles
length_summary = pd.Series(training_lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

# Show the length summary
length_summary

count    22524.000000
mean       308.161206
std        860.363748
min          1.000000
50%        153.000000
75%        328.250000
90%        645.000000
95%        942.850000
99%       1839.620000
max      45450.000000
dtype: float64

In [25]:
# Share of training emails longer than the 256-token limit
truncated_fraction = (training_lengths > MAX_LENGTH).mean()

# Tokens removed when emails are cut at the shared limit
discarded_tokens = np.maximum(training_lengths - MAX_LENGTH, 0).sum()

# Total tokens before any truncation
total_tokens = training_lengths.sum()

# Report how many emails and tokens the limit removes
print(f'Emails truncated: {truncated_fraction:.1%}')
print('Training tokens discarded:', f'{discarded_tokens / total_tokens:.1%}' if total_tokens > 0 else 'N/A')

Emails truncated: 32.2%
Training tokens discarded: 49.6%


In [26]:
# List each numeric label with its text from the training split
label_mapping = (splits['train'].select_columns(['label', 'label_text']).to_pandas().drop_duplicates().sort_values(['label', 'label_text']))

# Show the label meanings
label_mapping

,label,label_text
4,0,ham
0,1,spam


In [27]:
# Collect class counts for the development splits
class_summary = []

for split_name in ('train', 'validation'):
    # Count examples for each numeric label
    counts = pd.Series(splits[split_name]['label']).value_counts()

    for label in (0, 1):
        count = int(counts.get(label, 0))

        # Store the count and its share of the split
        class_summary.append({
            'split': split_name,
            'label': label,
            'count': count,
            'proportion': count / len(splits[split_name]),
            })

# Show class balance for train and validation
pd.DataFrame(class_summary)

,split,label,count,proportion
0,train,0,11681,0.518602
1,train,1,10843,0.481398
2,validation,0,2921,0.518643
3,validation,1,2711,0.481357


## 3. Save Prepared Data

The cleaned partitions, training vocabulary, and preprocessing configuration are saved locally for reuse across model experiments. Experiment notebooks load these artifacts instead of repeating the audit and preparation steps.

The test partition is stored for final evaluation but is not used to fit the vocabulary or select models.

In [28]:
# Store prepared artifacts under data/processed/enron
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'enron'

# Record the dataset, split, vocabulary, and cleaning choices
preprocessing_metadata = {
    'dataset_id': 'ham_spam-20,000',
    'dataset_revision': DATASET_REVISION,
    'split_seed': SPLIT_SEED,
    'validation_fraction': VALIDATION_FRACTION,
    'max_vocab_size': MAX_VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'max_length': MAX_LENGTH,
    'vocabulary_size': len(vocabulary),
    'label_mapping': {'0': 'ham', '1': 'spam'},
    'tokenizer': r'lowercase; \w+|[^\w\s]',
    'duplicate_policy': ('Normalize case and whitespace; keep first occurrence within ' 'each source split; remove train records overlapping test.'),
    'split_sizes': {name: len(dataset) for name, dataset in splits.items()}
    }

# Save the splits, vocabulary, and metadata for later experiments
save_prepared_data(splits = splits, vocabulary  = vocabulary, metadata = preprocessing_metadata, output_dir = PREPARED_DATA_DIR,)

# Confirm where the prepared data was written
print(f'Prepared data saved to: {PREPARED_DATA_DIR}')

Saving the dataset (1/1 shards): 100%|██████████| 1981/1981 [00:00<00:00, 31389.11 examples/s]


Prepared data saved to: C:\Users\Nouroz\Desktop\AI Machine Learning Semester Two\Advanced Deep Learning - AIGC-5500-0NA\Midtrem\AIGC5500-Midterm-Project\data\processed\enron


## 4. Second Dataset Preparation

In [29]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Load the CSV
df = pd.read_csv(PROJECT_ROOT / 'email_sample.csv')
df = df.dropna(subset=["text", "label"])
df = df.drop_duplicates(subset=["text"])
df['label'] = df['label'].map({"Ham": 0, "Spam": 1}).astype(int)

# Split into training and testing data
train_df, test_df = train_test_split(
    df,
    test_size=2000,
    random_state=SPLIT_SEED,
    stratify=df['label']
)

# Convert into a Hugging Face DatasetDict
raw_second_dataset = DatasetDict({
    'train': Dataset.from_pandas(
        train_df.reset_index(drop=True),
        preserve_index=False
    ),
    'test': Dataset.from_pandas(
        test_df.reset_index(drop=True),
        preserve_index=False
    )
})

print(raw_second_dataset)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'index', 'Unnamed: 0', 'label', 'text'],
        num_rows: 18000
    })
    test: Dataset({
        features: ['Unnamed: 0.1', 'index', 'Unnamed: 0', 'label', 'text'],
        num_rows: 2000
    })
})


In [30]:
# Inspect the dataset structure for training and test partitions
for split_name, split_data in raw_second_dataset.items():
    print(f'Partition: {split_name}')
    print(f'Rows: {len(split_data):,}')
    print(f'Columns: {split_data.column_names}')
    print(f'Features: {split_data.features}\n')

Partition: train
Rows: 18,000
Columns: ['Unnamed: 0.1', 'index', 'Unnamed: 0', 'label', 'text']
Features: {'Unnamed: 0.1': Value('int64'), 'index': Value('int64'), 'Unnamed: 0': Value('int64'), 'label': Value('int64'), 'text': Value('string')}

Partition: test
Rows: 2,000
Columns: ['Unnamed: 0.1', 'index', 'Unnamed: 0', 'label', 'text']
Features: {'Unnamed: 0.1': Value('int64'), 'index': Value('int64'), 'Unnamed: 0': Value('int64'), 'label': Value('int64'), 'text': Value('string')}



In [31]:
# Perform data quality inspection
quality_summary, shared_text_count = audit_dataset(raw_second_dataset)
quality_summary

,split,rows,missing_or_nonstring_text,empty_text,invalid_labels,duplicate_extra_rows,conflicting_label_groups
0,train,18000,0,0,0,21,0
1,test,2000,0,0,0,0,0


In [32]:
# Prepare and clean train and test splits
splits, cleaning_summary = prepare_splits(raw_second_dataset, split_seed = SPLIT_SEED, validation_fraction = 0.2)
cleaning_summary

,source_split,original_rows,invalid_or_empty_removed,duplicate_rows_removed,train_test_overlap_removed
0,train,18000,0,21,2
1,test,2000,0,0,0


In [33]:
# Observe split sizes 
split_sizes = pd.DataFrame([{'split': name, 'rows': len(dataset)}for name, dataset in splits.items()])
split_sizes

,split,rows
0,train,14381
1,validation,3596
2,test,2000


In [34]:
# Fit the vocabulary on training text only
vocabulary = build_vocabulary(splits['train']['text'], max_vocab_size = MAX_VOCAB_SIZE, min_frequency = MIN_FREQUENCY)

In [35]:
# Prepare development partitions. Test encoding is deferred.
encoded_datasets = {
    name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH) for name in ('train', 'validation')
    }

# Shuffle training batches with the first training seed
train_loader = make_data_loader(encoded_datasets['train'], batch_size = BATCH_SIZE, shuffle = True, seed = TRAINING_SEEDS[0])

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size=BATCH_SIZE, shuffle=False, seed=TRAINING_SEEDS[0],)

# Pull one validation batch to inspect the tensors
example_batch = next(iter(validation_loader))

# Check padded token, length, and label shapes
print('Token shape:', example_batch['tokens'].shape)
print('Length shape:', example_batch['lengths'].shape)
print('Label shape:', example_batch['labels'].shape)

# Lengths store the real token count before padding
print('First five lengths:', example_batch['lengths'][:5])

Token shape: torch.Size([64, 256])
Length shape: torch.Size([64])
Label shape: torch.Size([64])
First five lengths: tensor([145,  46, 132, 131,  48])


In [36]:
# Verify padding and token length
tokens = example_batch['tokens']
lengths = example_batch['lengths']
labels = example_batch['labels']

# Build a column index for every position in the batch
positions = torch.arange(tokens.shape[1]).unsqueeze(0)
# Mark positions at or past each real length as padding
padding_mask = positions >= lengths.unsqueeze(1)

# Padding positions must hold the padding ID
assert torch.all(tokens[padding_mask] == 0)
# Content positions must not use the padding ID
assert torch.all(tokens[~padding_mask] != 0)
# Token IDs must stay inside the vocabulary range
assert tokens.min().item() >= 0
assert tokens.max().item() < len(vocabulary)
# Labels must be binary
assert torch.all((labels == 0) | (labels == 1))

print('Batch padding, vocabulary IDs, and labels verified.')

Batch padding, vocabulary IDs, and labels verified.


In [37]:
# Count tokens in each training email before truncation
training_lengths = np.asarray(measure_token_lengths(splits['train']['text']))

# Summarize the length distribution, including the upper percentiles
length_summary = pd.Series(training_lengths).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

# Show the length summary
length_summary

count    14381.000000
mean       275.635074
std        745.422770
min          1.000000
50%        129.000000
75%        289.000000
90%        584.000000
95%        897.000000
99%       2201.400000
max      37463.000000
dtype: float64

In [38]:
# Share of training emails longer than the 256-token limit
truncated_fraction = (training_lengths > MAX_LENGTH).mean()

# Tokens removed when emails are cut at the shared limit
discarded_tokens = np.maximum(training_lengths - MAX_LENGTH, 0).sum()

# Total tokens before any truncation
total_tokens = training_lengths.sum()

# Report how many emails and tokens the limit removes
print(f'Emails truncated: {truncated_fraction:.1%}')
print('Training tokens discarded:', f'{discarded_tokens / total_tokens:.1%}' if total_tokens > 0 else 'N/A')

Emails truncated: 28.1%
Training tokens discarded: 48.4%


In [39]:
# Collect class counts for the development splits
class_summary = []

for split_name in ('train', 'validation'):
    # Count examples for each numeric label
    counts = pd.Series(splits[split_name]['label']).value_counts()

    for label in (0, 1):
        count = int(counts.get(label, 0))

        # Store the count and its share of the split
        class_summary.append({
            'split': split_name,
            'label': label,
            'count': count,
            'proportion': count / len(splits[split_name]),
            })

# Show class balance for train and validation
pd.DataFrame(class_summary)

,split,label,count,proportion
0,train,0,7646,0.531674
1,train,1,6735,0.468326
2,validation,0,1912,0.531702
3,validation,1,1684,0.468298


In [40]:
# Store prepared artifacts under data/processed/enron
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'kaggle'

# Record the dataset, split, vocabulary, and cleaning choices
preprocessing_metadata = {
    'dataset_id': 'kaggle',
    'dataset_revision': DATASET_REVISION,
    'split_seed': SPLIT_SEED,
    'validation_fraction': VALIDATION_FRACTION,
    'max_vocab_size': MAX_VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'max_length': MAX_LENGTH,
    'vocabulary_size': len(vocabulary),
    'label_mapping': {'0': 'ham', '1': 'spam'},
    'tokenizer': r'lowercase; \w+|[^\w\s]',
    'duplicate_policy': ('Normalize case and whitespace; keep first occurrence within ' 'each source split; remove train records overlapping test.'),
    'split_sizes': {name: len(dataset) for name, dataset in splits.items()}
    }

# Save the splits, vocabulary, and metadata for later experiments
save_prepared_data(splits = splits, vocabulary  = vocabulary, metadata = preprocessing_metadata, output_dir = PREPARED_DATA_DIR,)

# Confirm where the prepared data was written
print(f'Prepared data saved to: {PREPARED_DATA_DIR}')

Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 45143.97 examples/s]

Prepared data saved to: C:\Users\Nouroz\Desktop\AI Machine Learning Semester Two\Advanced Deep Learning - AIGC-5500-0NA\Midtrem\AIGC5500-Midterm-Project\data\processed\kaggle
